# Binance daily price and USDT-volume descriptive analysis

This notebook prepares and generates one Excel workbook for each of the 40 approved USDT-quoted pairs. The analysis uses immutable canonical daily CSV files, starts on `2019-07-01 00:00 UTC` or the pair's later first observed candle, and ends on `2026-09-01 00:00 UTC`. No observations are interpolated, forward-filled, backfilled, or synthesized.


In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys
import tempfile

import pandas as pd


def find_repository_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'pyproject.toml').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')


REPOSITORY_ROOT = find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / 'src'
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

from tcc_crypto.daily_price_volume_descriptive_analysis import (
    load_analysis_config,
    prepare_analysis_packets,
)

CONFIG_PATH = REPOSITORY_ROOT / 'configs/binance_daily_price_volume_descriptive_analysis.json'
PACKET_DIRECTORY = REPOSITORY_ROOT / 'tmp/daily_price_volume_workbook_inputs'
config, collection_config = load_analysis_config(CONFIG_PATH, repository_root=REPOSITORY_ROOT)
display(pd.DataFrame([
    ('Analysis ID', config.analysis_id),
    ('Pairs', len(collection_config.symbols)),
    ('Start (UTC)', config.analysis_start_utc.isoformat()),
    ('End (UTC)', config.analysis_end_utc.isoformat()),
    ('Output directory', str(config.output_directory)),
] , columns=['Field', 'Value']))


## Validate sources and prepare workbook inputs

The next cell validates timestamps, OHLC consistency, positivity, volume, coverage, and the approved time window before creating one JSON workbook input per pair.


In [ ]:
preparation_manifest = prepare_analysis_packets(
    CONFIG_PATH,
    repository_root=REPOSITORY_ROOT,
    packet_directory=PACKET_DIRECTORY,
)
display(preparation_manifest)


## Build the Excel workbooks

Workbook generation uses the bundled `@oai/artifact-tool`. Set `TCC_CRYPTO_NODE` and `TCC_CRYPTO_NODE_MODULES` to the Node executable and bundled module directory before running this cell. The builder is copied to a temporary directory so the repository does not receive a generated `node_modules` link.


In [ ]:
node_executable = os.environ.get('TCC_CRYPTO_NODE')
node_modules = os.environ.get('TCC_CRYPTO_NODE_MODULES')
if not node_executable or not node_modules:
    raise RuntimeError('Set TCC_CRYPTO_NODE and TCC_CRYPTO_NODE_MODULES before workbook generation.')

builder_source = REPOSITORY_ROOT / 'scripts/build_daily_price_volume_workbooks.mjs'
qa_directory = REPOSITORY_ROOT / 'tmp/daily_price_volume_workbook_qa'
with tempfile.TemporaryDirectory(prefix='tcc_crypto_artifact_tool_') as temporary_directory:
    temporary_path = Path(temporary_directory)
    builder_path = temporary_path / builder_source.name
    shutil.copy2(builder_source, builder_path)
    (temporary_path / 'node_modules').symlink_to(Path(node_modules), target_is_directory=True)
    subprocess.run(
        [node_executable, str(builder_path), str(PACKET_DIRECTORY), str(config.output_directory), str(qa_directory)],
        cwd=REPOSITORY_ROOT,
        check=True,
    )

final_manifest = pd.read_csv(config.output_directory / 'analysis_manifest.csv')
display(final_manifest)


## Interpretation

The workbooks describe non-stationary price levels and exchange-reported activity aggregated inside daily klines. They do not estimate returns, risk, or transaction-level behavior. Comparisons require caution because listing dates, observed spans, missing daily openings, and partial semesters differ.
